# 07 — Tableau Export

Builds a one-row-per-district file for the Tableau Public dashboard.

- Predictions come from the **same linear regression selected in notebook 05** (same features, same `train_test_split` with `random_state=42`, fit on the training set only). The coefficient check below confirms it matches.
- District names and states come from the raw CRDC Chronic Absenteeism file (`NCESLEAID` = `LEAID`, the same key used in notebook 02).
- `gap = grad_rate - predicted_grad_rate`. Positive = the district graduated more students than absenteeism and size alone would predict.
- `model_split` flags which districts the model was trained on. About 80% of predictions are in-sample, so the gap is a descriptive residual, not an out-of-sample forecast.
- Data is 2017-18 (CRDC/EDFacts). This is a methods demo, not a current ranking of districts.

Existing files are not modified; the output is a new file.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

## Refit the notebook 05 model

In [2]:
df = pd.read_csv('../data/processed/lea_features_v2.csv')

X = df[['absent_rate', 'log_enrolled']]
y = df['grad_rate']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

print('Intercept:', model.intercept_)
print('Coefficients:', model.coef_)
print('Test R²:', round(r2_score(y_test, model.predict(X_test)), 2))

Intercept: 95.63061201740912
Coefficients: [-0.62399045  0.26772666]
Test R²: 0.29


In [3]:
# Parity check against the values reported in notebook 05
assert np.isclose(model.intercept_, 95.63061201740912)
assert np.allclose(model.coef_, [-0.62399045, 0.26772666])
assert round(r2_score(y_test, model.predict(X_test)), 2) == 0.29
print('Matches notebook 05')

Matches notebook 05


## Predict for every district

In [4]:
df['predicted_grad_rate'] = model.predict(X)
df['gap'] = df['grad_rate'] - df['predicted_grad_rate']
df['model_split'] = np.where(df.index.isin(X_test.index), 'test', 'train')

print(df['model_split'].value_counts())
df[['grad_rate', 'predicted_grad_rate', 'gap']].describe().round(2)

model_split
train    4752
test     1188
Name: count, dtype: int64


,grad_rate,predicted_grad_rate,gap
count,5940.00,5940.00,5940.00
mean,88.12,88.12,-0.00
std,9.98,6.16,7.98
min,1.00,36.99,-83.46
25%,86.00,85.88,-2.47
50%,92.00,89.31,1.07
75%,95.00,92.04,3.97
max,99.00,97.93,30.81


## Add enrollment, state, and district name

In [5]:
base = pd.read_csv('../data/processed/lea_absenteeism_x_graduation_v2.csv')

# Same districts, same order, same absent_rate as the features file
assert (base['LEAID'].values == df['LEAID'].values).all()
assert np.allclose(base['absent_rate'], df['absent_rate'])

df = df.merge(base[['LEAID', 'enrolled']], on='LEAID', how='left', validate='1:1')

In [6]:
raw = pd.read_csv(
    '../data/raw/ID 814 SCH - Chronic Absenteeism.csv',
    usecols=['LEA_STATE', 'LEA_STATE_NAME', 'NCESLEAID', 'LEA_NAME'],
    dtype={'NCESLEAID': str}
)

# Raw file is school-level; keep one row per district
names = raw.drop_duplicates(subset=['NCESLEAID']).copy()
names['LEAID'] = pd.to_numeric(names['NCESLEAID'])

# Each LEAID should map to exactly one state and name
assert raw.drop_duplicates(['NCESLEAID', 'LEA_STATE', 'LEA_NAME'])['NCESLEAID'].is_unique

df = df.merge(names, on='LEAID', how='left', validate='1:1')
print('Districts missing a name:', df['LEA_NAME'].isna().sum())

Districts missing a name: 0


## Shape and save

In [7]:
out = pd.DataFrame({
    'LEAID': df['NCESLEAID'],  # 7-digit string keeps the leading zero; Tableau treats it as a dimension
    'state': df['LEA_STATE'],
    'state_name': df['LEA_STATE_NAME'].str.title(),
    'district_name': df['LEA_NAME'],
    'absent_rate': df['absent_rate'].round(2),
    'grad_rate': df['grad_rate'],
    'predicted_grad_rate': df['predicted_grad_rate'].round(2),
    'gap': df['gap'].round(2),
    'enrolled': df['enrolled'].astype(int),
    'model_split': df['model_split'],
})

assert len(out) == 5940
assert out['LEAID'].is_unique
assert out.notna().all().all()

out.head()

,LEAID,state,state_name,district_name,absent_rate,grad_rate,predicted_grad_rate,gap,enrolled,model_split
0,0100005,AL,Alabama,Albertville City,13.43,94.0,89.55,4.45,5444,train
1,0100006,AL,Alabama,Marshall County,16.29,91.0,87.78,3.22,5698,train
2,0100007,AL,Alabama,Hoover City,9.26,94.0,92.42,1.58,14369,train
3,0100008,AL,Alabama,Madison City,9.87,96.0,91.96,4.04,10804,train
4,0100011,AL,Alabama,Leeds City,11.89,95.0,90.25,4.75,2027,train


In [8]:
output_path = '../data/processed/tableau_district_predictions_2017_18.csv'
out.to_csv(output_path, index=False)
print(f'Saved {len(out)} rows to {output_path}')

Saved 5940 rows to ../data/processed/tableau_district_predictions_2017_18.csv
